# Ecuación de calor 1D: solución

Implementación completa del método FTCS con comparación frente a la solución analítica.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def u0(x):
    return np.sin(np.pi * x)


def exact_solution(x, t, alpha=1.0):
    return np.exp(-alpha * np.pi**2 * t) * np.sin(np.pi * x)


def build_grid(alpha=1.0, L=1.0, T=0.12, N=80, r_obj=0.45):
    dx = L / N
    dt = r_obj * dx * dx / alpha
    nt = int(np.ceil(T / dt))
    dt = T / nt
    r = alpha * dt / (dx * dx)
    x = np.linspace(dx, L - dx, N - 1)
    return x, dx, dt, nt, r


def ftcs_step(u, r):
    un = u.copy()
    u[1:-1] = un[1:-1] + r * (un[2:] - 2.0 * un[1:-1] + un[:-2])
    u[0] = 0.0
    u[-1] = 0.0
    return u


def solve_heat(alpha=1.0, L=1.0, T=0.12, N=80, r_obj=0.45, store_every=6):
    x_in, dx, dt, nt, r = build_grid(alpha=alpha, L=L, T=T, N=N, r_obj=r_obj)
    x = np.concatenate(([0.0], x_in, [L]))
    u = np.zeros_like(x)
    u[1:-1] = u0(x_in)

    t_hist = [0.0]
    U_hist = [u.copy()]

    for n in range(nt):
        u = ftcs_step(u, r)
        t_next = (n + 1) * dt
        if (n + 1) % store_every == 0 or (n + 1) == nt:
            t_hist.append(t_next)
            U_hist.append(u.copy())

    return x, u, nt * dt, r, np.array(t_hist), np.array(U_hist)


alpha = 1.0
L = 1.0
T = 0.12
N = 80
r_obj = 0.45
x, u_num, t_final, r, t_hist, U_hist = solve_heat(alpha=alpha, L=L, T=T, N=N, r_obj=r_obj, store_every=4)
u_ex = exact_solution(x, t_final, alpha=alpha)
err_l2 = np.sqrt(np.mean((u_num - u_ex) ** 2))
print(f"r={r:.6f}")
print(f"t_final={t_final:.6f}")
print(f"error L2={err_l2:.3e}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, u_num, lw=2.2, label="Numérica")
plt.plot(x, u_ex, "--", lw=2.0, label="Exacta")
plt.title("Ecuación de calor 1D en t_final")
plt.xlabel("x")
plt.ylabel("u(x,t_final)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## Evolución temporal en varios instantes


In [ ]:
plt.figure(figsize=(8, 4))
for t_ref in [0.00, 0.03, 0.06, 0.09, 0.12]:
    idx = int(np.argmin(np.abs(t_hist - t_ref)))
    plt.plot(x, U_hist[idx], lw=1.8, label=f"t={t_hist[idx]:.3f}")
plt.title("Calor 1D: perfiles temporales")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## Barrido de resolución espacial


In [ ]:
N_vals = [40, 80, 120, 160]
errs = []
for n_case in N_vals:
    x_c, u_c, t_c, r_c, _, _ = solve_heat(alpha=alpha, L=L, T=T, N=n_case, r_obj=r_obj, store_every=10)
    u_ex_c = exact_solution(x_c, t_c, alpha=alpha)
    errs.append(np.sqrt(np.mean((u_c - u_ex_c) ** 2)))

plt.figure(figsize=(7.8, 4))
plt.plot(N_vals, errs, "o-", lw=2.0)
plt.title("Calor 1D: error L2 vs N")
plt.xlabel("N")
plt.ylabel("Error L2")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
